In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download(
    "sriramr/fruits-fresh-and-rotten-for-classification"
)

Using Colab cache for faster access to the 'fruits-fresh-and-rotten-for-classification' dataset.


In [2]:
import tensorflow as tf
import os
train_path = "/kaggle/input/fruits-fresh-and-rotten-for-classification/dataset/train"
test_path = "/kaggle/input/fruits-fresh-and-rotten-for-classification/dataset/test"

In [3]:
validation_path = os.path.join(path, 'dataset', 'validation')
test_path = os.path.join(path, 'dataset', 'test')

In [4]:
img_height = 224
img_width = 224
batch_size = 32

In [5]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

Found 10901 files belonging to 6 classes.
Using 8721 files for training.
Found 10901 files belonging to 6 classes.
Using 2180 files for validation.


In [6]:
original_class_name_for_display = train_ds.class_names

In [7]:
def normalize(image, label):
    image = tf.cast(image / 255.0, tf.float32)
    return image, label

In [8]:
train_ds = train_ds.map(normalize)
val_ds = val_ds.map(normalize)

In [9]:
print("Image dataset loaded and normalized successfully.")
print(f"Number of batches in normalized training dataset: {tf.data.experimental.cardinality(train_ds).numpy()}")
print(f"Number of batches in normalized validation dataset: {tf.data.experimental.cardinality(val_ds).numpy()}")
print(f"Image size: {img_height}x{img_width}")
print(f"Batch size: {batch_size}")
print(f"Original Classes found: {original_class_name_for_display}")

Image dataset loaded and normalized successfully.
Number of batches in normalized training dataset: 273
Number of batches in normalized validation dataset: 69
Image size: 224x224
Batch size: 32
Original Classes found: ['freshapples', 'freshbanana', 'freshoranges', 'rottenapples', 'rottenbanana', 'rottenoranges']


In [10]:
def map_binary(image,label):
  binary_label = tf.cast(label>= len(original_class_name_for_display)//2,tf.float32)
  return image,binary_label

In [11]:
train_ds = train_ds.map(map_binary)
val_ds = val_ds.map(map_binary)

In [12]:
data_augumentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
])

def augument(image,label):
  image = data_augumentation(image)
  return image,label

train_ds_augument = train_ds.map(augument)

In [13]:
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(img_height, img_width, 3)),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [14]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,089 (42.61 MB)

 Trainable params: 11,169,089 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [16]:
Epoch = 20

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    train_ds_augument,
    validation_data=val_ds,
    epochs=Epoch,
    callbacks=[early_stopping]
)

Epoch 1/20
273/273 ━━━━━━━━━━━━━━━━━━━━ 158s 547ms/step - accuracy: 0.7894 - loss: 0.4500 - val_accuracy: 0.9133 - val_loss: 0.2321
Epoch 2/20
273/273 ━━━━━━━━━━━━━━━━━━━━ 137s 501ms/step - accuracy: 0.9010 - loss: 0.2493 - val_accuracy: 0.9271 - val_loss: 0.2092
Epoch 3/20
273/273 ━━━━━━━━━━━━━━━━━━━━ 135s 492ms/step - accuracy: 0.9208 - loss: 0.2062 - val_accuracy: 0.9344 - val_loss: 0.1766
Epoch 4/20
273/273 ━━━━━━━━━━━━━━━━━━━━ 135s 495ms/step - accuracy: 0.9211 - loss: 0.2013 - val_accuracy: 0.9289 - val_loss: 0.1816
Epoch 5/20
273/273 ━━━━━━━━━━━━━━━━━━━━ 142s 494ms/step - accuracy: 0.9341 - loss: 0.1641 - val_accuracy: 0.9514 - val_loss: 0.1272
Epoch 6/20
273/273 ━━━━━━━━━━━━━━━━━━━━ 137s 502ms/step - accuracy: 0.9409 - loss: 0.1504 - val_accuracy: 0.9555 - val_loss: 0.1092
Epoch 7/20
273/273 ━━━━━━━━━━━━━━━━━━━━ 137s 501ms/step - accuracy: 0.9473 - loss: 0.1261 - val_accuracy: 0.9596 - val_loss: 0.1080
Epoch 8/20
273/273 ━━━━━━━━━━━━━━━━━━━━ 136s 495ms/step - accuracy: 0.9463 -

In [17]:
test_path = os.path.join(path, 'dataset', 'test')
test_ds = tf.keras.utils.image_dataset_from_directory(
    test_path,
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    shuffle=False
)

Found 2698 files belonging to 6 classes.


In [18]:
test_ds = test_ds.map(normalize)
test_ds = test_ds.map(map_binary)

In [21]:
model.save("fruit_classifier.keras")
# Save complete model in H5 format
model.save("fruit_classifier.h5")

print("Model saved successfully!")

Model saved successfully!


In [20]:
from IPython.testing import test
loss , accuracy = model.evaluate(test_ds)
test_loss = loss
test_accuracy = accuracy
print(f"Test Loss: {test_loss}")
print(f"Test Accuracy: {test_accuracy}")

85/85 ━━━━━━━━━━━━━━━━━━━━ 14s 168ms/step - accuracy: 0.9863 - loss: 0.0453
Test Loss: 0.04527239874005318
Test Accuracy: 0.9862861633300781


In [25]:
# Save model in Keras format
model.save("fruit_classifier.keras")

# Save class names
import pickle

class_names = [
    'freshapples',
    'freshbanana',
    'freshoranges',
    'rottenapples',
    'rottenbanana',
    'rottenoranges'
]

with open("class_names.pkl", "wb") as f:
    pickle.dump(class_names, f)

# Download files (Colab)
from google.colab import files

files.download("fruit_classifier.keras")
files.download("class_names.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>